# Módulo 03 · `groupby` a fondo

**Curso de pandas intermedio → avanzado** · Dataset: *Olist* (muestras en `data/`)

## Objetivos
1. Pensar en **split → apply → combine** y saber qué devuelve cada operación.
2. Escribir agregaciones claras con **agregaciones con nombre** (*named aggregation*).
3. Controlar llaves múltiples, `as_index`, `observed`, `dropna` y `sort`.
4. Usar **`transform`** para calcular valores de grupo sin perder filas.
5. Usar **`filter`** para quedarte con grupos completos que cumplen una condición.
6. Saber **cuándo usar `apply`** (y cuándo no), y los cambios de pandas 3.
7. Agrupar por tiempo con `pd.Grouper` y elegir la opción más rápida.

| Sección | Tema |
|---|---|
| 0 | Setup y carga |
| 1 | El objeto `GroupBy` |
| 2 | `agg` y agregaciones con nombre |
| 3 | Llaves múltiples y parámetros del `groupby` |
| 4 | `transform` |
| 5 | `filter` |
| 6 | `apply` y top‑N por grupo |
| 7 | Agrupar por tiempo con `pd.Grouper` |
| 8 | Rendimiento |
| ✍️ | Ejercicios · ✅ Soluciones |

### Mapa mental: ¿qué forma tiene el resultado?

| Operación | Filas del resultado | Uso típico |
|---|---|---|
| `agg` / `sum` / `mean`… | **una por grupo** | resúmenes |
| `transform` | **las mismas que la entrada** | % del grupo, normalizar, rellenar |
| `filter` | **subconjunto** de las filas originales | quedarse con grupos que cumplen algo |
| `apply` | depende de lo que devuelva tu función | lo que no cabe en lo anterior |

## 0 · Setup y carga

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

DATA = Path("data")
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)


def pendiente(*valores) -> bool:
  # True si algún valor sigue siendo `...` (ejercicio sin resolver)
  return any(v is ... for v in valores)


print("pandas", pd.__version__)

pandas 3.1.1


In [2]:
orders = pd.read_csv(
  DATA / "olist_orders_dataset.csv",
  usecols=["order_id", "customer_id", "order_status", "order_purchase_timestamp",
           "order_delivered_customer_date", "order_estimated_delivery_date"],
  dtype={"order_status": "category"},
  parse_dates=["order_purchase_timestamp", "order_delivered_customer_date",
               "order_estimated_delivery_date"],
)

items = pd.read_csv(
  DATA / "olist_order_items_dataset.csv",
  usecols=["order_id", "order_item_id", "product_id", "seller_id", "price", "freight_value"],
  dtype={"order_item_id": "int8"},
)

payments = pd.read_csv(
  DATA / "olist_order_payments_dataset.csv",
  dtype={"payment_type": "category", "payment_sequential": "int8",
         "payment_installments": "int8"},
)

products = pd.read_csv(
  DATA / "olist_products_dataset.csv",
  usecols=["product_id", "product_category_name", "product_weight_g"],
)

traduccion = pd.read_csv(DATA / "product_category_name_translation.csv")

customers = pd.read_csv(
  DATA / "olist_customers_dataset.csv",
  usecols=["customer_id", "customer_state"],
  dtype={"customer_state": "category"},
)

Enriquecemos `items` con la categoría en inglés usando *lookups* con `map` (Módulo 02). Dejamos los faltantes como `NaN` a propósito: los usaremos para hablar de `dropna`.

In [3]:
cat_en = traduccion.set_index("product_category_name")["product_category_name_english"]

items = items.assign(
  categoria=items["product_id"]
      .map(products.set_index("product_id")["product_category_name"])
      .map(cat_en),
  total=items["price"] + items["freight_value"],
)
items.head()

,order_id,order_item_id,product_id,seller_id,price,freight_value,categoria,total
0,008a40c2b8b21893e04945ce58854b55,1,016711f78a6a87696645db0a6a834826,a5cba26a62b8b4d0145b68b841e62e7f,99.00,13.00,health_beauty,112.00
1,00fc308dd7f0937682698becaa9dcc45,1,940d38f0ef286bd93612fcdc91dd38ed,53e4c6e0f4312d4d2107a8c9cddf45cd,85.71,18.40,housewares,104.11
2,0124e0c5bfae1979838a91e47df53f14,1,0bcc3eeca39e1064258aa1e932269894,1f50f920176fa81dab994f9023523100,49.90,12.48,garden_tools,62.38
3,042649285cd08ef7c546445419b43407,1,927225847efa8853ebf52c9b5b9b8cc6,855668e0971d4dfd7bef1b6a4133b41b,59.99,18.35,baby,78.34
4,0452166f3b0d8e378179ecea1efdfe05,1,40df4be4429553e18755913ac3cfeac5,ff1e15b778c700abdd4d239b81ac466d,59.90,11.91,musical_instruments,71.81


## 1 · El objeto `GroupBy`

`df.groupby(...)` **no calcula nada** todavía: crea un objeto que sabe cómo partir los datos. El cálculo ocurre cuando pides una operación.

In [4]:
g = items.groupby("seller_id")
print(type(g).__name__)
print("Grupos:", g.ngroups)

DataFrameGroupBy
Grupos: 212


In [5]:
# Mirar un grupo concreto
vendedor = items["seller_id"].value_counts().idxmax()
g.get_group(vendedor)

,order_id,order_item_id,product_id,seller_id,price,freight_value,categoria,total
2,0124e0c5bfae1979838a91e47df53f14,1,0bcc3eeca39e1064258aa1e932269894,1f50f920176fa81dab994f9023523100,49.9,12.48,garden_tools,62.38
11,07bebe0626c8053ad425381fe0882655,1,53759a2ecddad2bb87a079a1f1519f73,1f50f920176fa81dab994f9023523100,59.9,17.67,garden_tools,77.57
57,2c5871f32e23db4f41bf1c58df2f1bee,1,389d119b48cf3043d311335e499d9c6b,1f50f920176fa81dab994f9023523100,59.9,13.44,garden_tools,73.34
68,338be6220254203898339bb8b6021138,1,389d119b48cf3043d311335e499d9c6b,1f50f920176fa81dab994f9023523100,53.9,8.11,garden_tools,62.01
69,338be6220254203898339bb8b6021138,2,389d119b48cf3043d311335e499d9c6b,1f50f920176fa81dab994f9023523100,53.9,8.11,garden_tools,62.01
73,3762c0ef92b5bc397671e6d7e6d5672f,1,422879e10f46682990de24d770e7f83d,1f50f920176fa81dab994f9023523100,59.9,13.44,garden_tools,73.34
104,4abd25ff34f0b324cadc39a6529b05c8,1,4b5df063d69ffafb87c279672ecb4191,1f50f920176fa81dab994f9023523100,49.9,32.85,baby,82.75
166,778d960ed98fe278fca42ee20d002b1f,1,53759a2ecddad2bb87a079a1f1519f73,1f50f920176fa81dab994f9023523100,53.9,10.13,garden_tools,64.03
202,99f10393ec3328b3b416a154afd5d714,1,53759a2ecddad2bb87a079a1f1519f73,1f50f920176fa81dab994f9023523100,59.9,17.67,garden_tools,77.57
203,99f10393ec3328b3b416a154afd5d714,2,53759a2ecddad2bb87a079a1f1519f73,1f50f920176fa81dab994f9023523100,59.9,17.67,garden_tools,77.57


### `size` vs `count`

- `size()` cuenta **filas** por grupo (incluye nulos).
- `count()` cuenta **valores no nulos** por columna.

In [6]:
gc = items.groupby("categoria", dropna=False)
pd.DataFrame({"size": gc.size(), "count_categoria": gc["categoria"].count()}).tail(4)

,size,count_categoria
categoria,,
telephony,17,17
toys,7,7
watches_gifts,15,15
NaN,6,0


Observa la fila `NaN`: `size` cuenta 6 filas sin categoría, mientras `count` de la propia columna da 0 porque todos sus valores son nulos.

### Iterar sobre grupos

Funciona, pero úsalo para **inspeccionar**, no para calcular: casi siempre hay una forma vectorizada.

In [7]:
for estado, grupo in orders.merge(customers, on="customer_id").groupby("customer_state", observed=True):
  if len(grupo) >= 30:
    print(f"{estado}: {len(grupo)} pedidos")

MG: 32 pedidos
RJ: 40 pedidos
SP: 130 pedidos


## 2 · `agg` y agregaciones con nombre

Hay varias formas de llamar a `agg`. De menos a más explícita:

In [8]:
# 1) Una función sobre una columna
items.groupby("categoria")["price"].mean().sort_values(ascending=False).head(3)

categoria
consoles_games       399.000000
home_appliances_2    340.826667
music                299.000000
Name: price, dtype: float64

In [9]:
# 2) Varias funciones: produce columnas MultiIndex si hay varias columnas
items.groupby("categoria")[["price", "freight_value"]].agg(["mean", "max"]).head(3)

price        freight_value       
                        mean    max          mean    max
categoria                                               
air_conditioning  279.450000  369.9       21.2050  24.41
audio             137.200000  137.2       25.0900  25.09
auto              167.476667  415.7       21.6275  51.03

In [10]:
# 3) Diccionario columna -> función(es)
items.groupby("categoria").agg({"price": "sum", "freight_value": ["mean", "max"]}).head(3)

price freight_value       
                      sum          mean    max
categoria                                     
air_conditioning   558.90       21.2050  24.41
audio              137.20       25.0900  25.09
auto              2009.72       21.6275  51.03

### ✅ Agregaciones con nombre (*named aggregation*)

La forma recomendada: **tú eliges el nombre de cada columna de salida**, sin `MultiIndex` y sin renombrar después.

```python
df.groupby(llave).agg(
    nombre_salida=("columna", "funcion"),
    ...
)
```

In [11]:
por_categoria = items.groupby("categoria").agg(
  n_items=("order_id", "size"),
  n_pedidos=("order_id", "nunique"),
  ingreso=("price", "sum"),
  precio_medio=("price", "mean"),
  precio_max=("price", "max"),
  flete_medio=("freight_value", "mean"),
)
por_categoria.sort_values("ingreso", ascending=False).head()

,n_items,n_pedidos,ingreso,precio_medio,precio_max,flete_medio
categoria,,,,,,
watches_gifts,15,14,3308.70,220.580000,1700.00,14.123333
sports_leisure,34,30,3143.27,92.449118,339.00,18.668235
health_beauty,29,26,2715.67,93.643793,349.90,15.496897
bed_bath_table,33,26,2635.26,79.856364,205.00,18.317273
computers_accessories,27,25,2524.73,93.508519,327.49,18.848148


### Funciones propias

Puedes pasar cualquier función que reciba una Series y devuelva un escalar. Las funciones **con nombre** dejan el código más legible que las lambdas largas.

In [12]:
def p90(s: pd.Series) -> float:
  return s.quantile(0.9)


def rango(s: pd.Series) -> float:
  return s.max() - s.min()


items.groupby("categoria").agg(
  precio_p90=("price", p90),
  precio_rango=("price", rango),
  pct_flete_gratis=("freight_value", lambda s: (s == 0).mean() * 100),
).sort_values("precio_p90", ascending=False).head()

,precio_p90,precio_rango,pct_flete_gratis
categoria,,,
home_appliances,756.370,1028.00,0.0
home_appliances_2,471.998,292.51,0.0
consoles_games,399.000,0.00,0.0
musical_instruments,358.670,337.89,0.0
auto,357.099,387.80,0.0


> 💡 Las funciones *built-in* pasadas como string (`"sum"`, `"mean"`, `"nunique"`, `"first"`…) están optimizadas en C. Una función propia se ejecuta **una vez por grupo en Python**: es más lenta (sección 8).

## 3 · Llaves múltiples y parámetros del `groupby`

| Parámetro | Default (pandas 3) | Qué controla |
|---|---|---|
| `as_index` | `True` | si las llaves quedan en el índice o como columnas |
| `sort` | `True` | ordenar por las llaves |
| `dropna` | `True` | **descarta** grupos cuya llave es `NaN` |
| `observed` | `True` | con llaves `category`, sólo combinaciones que existen |

In [13]:
pagos_estado = (
  payments
  .merge(orders[["order_id", "customer_id"]], on="order_id")
  .merge(customers, on="customer_id")
)

pagos_estado.groupby(["customer_state", "payment_type"]).agg(
  n=("order_id", "size"),
  monto=("payment_value", "sum"),
).head(8)

n    monto
customer_state payment_type            
AC             boleto        1   158.27
AL             credit_card   1   162.29
BA             boleto        3   196.29
               credit_card   8  1761.85
               voucher       1    50.00
CE             boleto        2   197.54
               credit_card   1   244.27
DF             boleto        2   351.58

In [14]:
# as_index=False: llaves como columnas (cómodo para exportar o graficar)
pagos_estado.groupby(["customer_state", "payment_type"], as_index=False).agg(
  monto=("payment_value", "sum")
).head(4)

,customer_state,payment_type,monto
0,AC,boleto,158.27
1,AL,credit_card,162.29
2,BA,boleto,196.29
3,BA,credit_card,1761.85


### `observed` con categorías

`customer_state` y `payment_type` son `category`. Con `observed=False` pandas generaría **todas** las combinaciones posibles (cada estado × cada tipo de pago), aunque no existan en los datos:

In [15]:
for obs in (True, False):
  n = pagos_estado.groupby(["customer_state", "payment_type"], observed=obs).size()
  print(f"observed={obs!s:5} -> {len(n)} filas, ceros: {(n == 0).sum()}")

observed=True  -> 39 filas, ceros: 0
observed=False -> 88 filas, ceros: 49


### `dropna`: cuidado con las llaves faltantes

Por defecto, los ítems **sin categoría desaparecen** del resultado. Si sumas, el total no cuadra con el total real:

In [16]:
total_real = items["price"].sum()
total_grupos = items.groupby("categoria")["price"].sum().sum()
total_grupos_na = items.groupby("categoria", dropna=False)["price"].sum().sum()

print(f"Total real        : {total_real:,.2f}")
print(f"Suma por grupos   : {total_grupos:,.2f}   <- ¡faltan los NaN!")
print(f"Con dropna=False  : {total_grupos_na:,.2f}")

Total real        : 36,599.36
Suma por grupos   : 36,020.73   <- ¡faltan los NaN!
Con dropna=False  : 36,599.36


> ⚠️ Antes de reportar totales por grupo, verifica que **cuadren** con el total general. `dropna=False` o rellenar la llave (`fillna("sin_categoria")`) evita sorpresas.

## 4 · `transform`

`transform` calcula algo **por grupo** pero devuelve un resultado **alineado con las filas originales**. Es la herramienta para "comparar cada fila con su grupo".

In [17]:
ej = items.loc[items["order_id"].duplicated(keep=False), ["order_id", "order_item_id", "price"]].head(7)

ej.assign(
  total_pedido=ej.groupby("order_id")["price"].transform("sum"),
  n_items=ej.groupby("order_id")["price"].transform("size"),
)

,order_id,order_item_id,price,total_pedido,n_items
62,2fc4ba497db1306b9048aceaf0f7e5a9,1,172.33,284.66,2
63,2fc4ba497db1306b9048aceaf0f7e5a9,2,112.33,284.66,2
68,338be6220254203898339bb8b6021138,1,53.90,107.80,2
69,338be6220254203898339bb8b6021138,2,53.90,107.80,2
87,3caf6c286c8cde37ea9c0fb2da0e3169,1,7.90,23.70,3
88,3caf6c286c8cde37ea9c0fb2da0e3169,2,7.90,23.70,3
89,3caf6c286c8cde37ea9c0fb2da0e3169,3,7.90,23.70,3


### Patrones típicos con `transform`

In [18]:
gp = items.groupby("categoria")["price"]

items_t = items.assign(
  # % del ingreso de su categoría
  pct_de_categoria=items["price"] / gp.transform("sum") * 100,
  # desviación respecto a la media de su categoría (z-score)
  z_precio=(items["price"] - gp.transform("mean")) / gp.transform("std"),
  # ranking dentro de la categoría (1 = más caro)
  rank_en_categoria=gp.rank(method="dense", ascending=False),
  # ¿es el ítem más caro de su categoría?
  es_max=items["price"].eq(gp.transform("max")),
)
items_t.loc[items_t["categoria"] == "health_beauty",
            ["price", "pct_de_categoria", "z_precio", "rank_en_categoria", "es_max"]].sort_values("rank_en_categoria").head()

,price,pct_de_categoria,z_precio,rank_en_categoria,es_max
257,349.90,12.884482,3.570727,1.0,True
6,201.34,7.414008,1.500661,2.0,False
134,200.00,7.364665,1.481989,3.0,False
184,163.00,6.002202,0.966424,4.0,False
241,144.00,5.302559,0.701674,5.0,False


Algunos métodos ya devuelven el tamaño original sin necesidad de `transform`: `rank`, `cumsum`, `cumcount`, `shift`, `diff`, `pct_change`, `fillna` (vía `transform`) …

In [19]:
# Rellenar faltantes con la mediana de su grupo
peso_cat = products.assign(categoria=products["product_category_name"].map(cat_en))
mediana_cat = peso_cat.groupby("categoria")["product_weight_g"].transform("median")
peso_relleno = peso_cat["product_weight_g"].fillna(mediana_cat)
print("Nulos antes:", peso_cat["product_weight_g"].isna().sum(), "| después:", peso_relleno.isna().sum())

Nulos antes: 0 | después: 0


In [20]:
# Secuencias dentro de un grupo: numerar compras y calcular tiempo entre pedidos de cada estado
ord_estado = (orders.merge(customers, on="customer_id")
                    .sort_values("order_purchase_timestamp"))

g_est = ord_estado.groupby("customer_state", observed=True)
ord_estado = ord_estado.assign(
  n_pedido_estado=g_est.cumcount() + 1,
  dias_desde_anterior=g_est["order_purchase_timestamp"].diff().dt.days,
)
ord_estado.loc[ord_estado["customer_state"] == "RJ",
               ["order_purchase_timestamp", "n_pedido_estado", "dias_desde_anterior"]].head()

,order_purchase_timestamp,n_pedido_estado,dias_desde_anterior
159,2017-01-28 20:09:18,1,NaN
217,2017-02-07 10:26:05,2,9.0
203,2017-02-08 14:36:40,3,1.0
291,2017-02-08 14:42:14,4,0.0
215,2017-03-02 07:28:36,5,21.0


## 5 · `filter`

`filter` evalúa una condición **por grupo** (la función recibe el sub-DataFrame y devuelve `True`/`False`) y conserva **todas las filas** de los grupos que la cumplen.

In [21]:
# Ítems de vendedores con al menos 3 ventas
items_vend_activos = items.groupby("seller_id").filter(lambda d: len(d) >= 3)
print("Ítems:", len(items), "->", len(items_vend_activos))
print("Vendedores:", items["seller_id"].nunique(), "->", items_vend_activos["seller_id"].nunique())

Ítems: 336 -> 107
Vendedores: 212 -> 21


La misma idea con `transform` suele ser **más rápida** (no llama a una función Python por grupo) y te deja combinar condiciones:

In [22]:
n_por_vendedor = items.groupby("seller_id")["order_id"].transform("size")
alt = items[n_por_vendedor >= 3]
assert alt.equals(items_vend_activos)

## 6 · `apply` y top‑N por grupo

`apply` le pasa **cada sub-DataFrame** a tu función y combina lo que devuelva:

- un escalar → Series (como `agg`)
- una Series → DataFrame (una fila por grupo)
- un DataFrame → se concatenan

### Cambio en pandas 3

En pandas 3, `apply` **ya no incluye las columnas de agrupación** dentro del sub-DataFrame que recibe tu función (antes sí, con un warning). Si la necesitas, está en el índice del resultado o puedes leerla con `d.name`.

In [23]:
def resumen_pedido(d: pd.DataFrame) -> pd.Series:
  return pd.Series({
    "columnas_visibles": ", ".join(d.columns[:3]) + ", ...",
    "n_items": len(d),
    "item_mas_caro": d.loc[d["price"].idxmax(), "categoria"],
  })

items.groupby("order_id").apply(resumen_pedido).sort_values("n_items", ascending=False).head(3)

,columnas_visibles,n_items,item_mas_caro
order_id,,,
3caf6c286c8cde37ea9c0fb2da0e3169,"order_item_id, product_id, seller_id, ...",6,bed_bath_table
f97b2bdfbaca9c2f04722d8cbc8cc8a5,"order_item_id, product_id, seller_id, ...",4,sports_leisure
5cc6d864fa9e1a90eaf5207737549ffc,"order_item_id, product_id, seller_id, ...",3,garden_tools


Fíjate que `order_id` **no** aparece en `columnas_visibles`.

### Top‑N por grupo: tres formas

Los ítems más caros de cada categoría.

In [24]:
# a) sort + groupby.head  -> rápido y conserva el índice original  ✅
top_a = items.sort_values("price", ascending=False).groupby("categoria").head(2)

# b) apply + nlargest     -> legible, más lento
top_b = items.groupby("categoria").apply(lambda d: d.nlargest(2, "price"))

# c) rank + máscara        -> útil si además quieres el ranking
top_c = items[items.groupby("categoria")["price"].rank(method="first", ascending=False) <= 2]

print(len(top_a), len(top_b), len(top_c))
top_b.head(6)

67 67 67


order_id  order_item_id                        product_id                         seller_id  \
categoria                                                                                                                                   
air_conditioning 9    0739104e30e20de6c0956c0889a9b9dc              1  2f07e976774b91520e17dd3b2dba73c3  11742f81550ac064cc9a40f67151d9ad   
                 211  a52e6fa2d7c6ecaf73ff5f7f10438339              1  28fd1639ff52d870a3d2c38090d2d902  2199e7fe213c16213bf5d6a7eadc9a5d   
audio            308  f00c74caaa25430b1fa361c158e8996a              1  fb10a064e6f5151ebfffc589e166c917  20b54c376b794ed028df09a3cd88e8dc   
auto             16   0bc16cf0c58995bbc893af2e151534d0              1  8e48c45980a3fee42f12f7986a4b68d5  e5e33c0d8e7e002f398f5dc4cbbb338f   
                 209  a25b9ed9a3a53f53489c105482625129              1  3cf8d7db7dbaeb28f077151adfc59340  c12b92bf1c350f3e6bf88d3596219c86   
baby             117  51879b7474407f56048b6c36cc3e263c              1  581de99ffa609d5f7b841f0d81df3f2f  e8b4225284fbb02d16f200513f1f395d   

                       price  freight_value   total  
categoria                                            
air_conditioning 9    369.90          24.41  394.31  
                 211  189.00          18.00  207.00  
audio            308  137.20          25.09  162.29  
auto             16   415.70          20.79  436.49  
                 209  369.00          16.16  385.16  
baby             117  185.49          37.38  222.87

Observa que `top_b` trae un `MultiIndex` (categoría, índice original) y, por el cambio de pandas 3, **ya no contiene** la columna `categoria`: está sólo en el índice. Con `group_keys=False` evitas el nivel extra.

> 🎯 **Regla práctica:** intenta primero `agg` → `transform` → métodos de grupo (`head`, `nth`, `rank`, `cumsum`…). Usa `apply` sólo cuando tu lógica necesite **varias columnas a la vez** y no exista una alternativa vectorizada.

## 7 · Agrupar por tiempo con `pd.Grouper`

`pd.Grouper(key=columna, freq=...)` agrupa una columna de fechas en periodos **sin crear columnas auxiliares** y puede combinarse con otras llaves. Las frecuencias más comunes: `"D"`, `"W"`, `"ME"` (fin de mes), `"QE"` (fin de trimestre), `"YE"`.

In [25]:
mensual = orders.groupby(pd.Grouper(key="order_purchase_timestamp", freq="ME")).agg(
  pedidos=("order_id", "size"),
  entregados=("order_delivered_customer_date", "count"),
)
mensual.tail(6)

,pedidos,entregados
order_purchase_timestamp,,
2018-03-31,23,23
2018-04-30,21,21
2018-05-31,21,21
2018-06-30,25,25
2018-07-31,15,15
2018-08-31,21,20


In [26]:
# Combinado con otra llave
orders.groupby([pd.Grouper(key="order_purchase_timestamp", freq="QE"), "order_status"], observed=True).size().tail(6)

order_purchase_timestamp  order_status
2018-03-31                canceled         1
                          delivered       79
                          shipped          1
2018-06-30                delivered       67
2018-09-30                canceled         1
                          delivered       35
dtype: int64

> Profundizaremos en `resample` y ventanas en el Módulo 06.

## 8 · Rendimiento

Comparemos tres formas de calcular la media de precio por vendedor. Repetimos los datos para que la diferencia se note.

In [27]:
grande = pd.concat([items] * 300, ignore_index=True)
print(f"{len(grande):,} filas")
gg = grande.groupby("seller_id")["price"]

100,800 filas


In [28]:
%timeit gg.mean()

504 μs ± 22.3 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [29]:
%timeit gg.agg(lambda s: s.mean())

3.93 ms ± 311 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [30]:
%timeit gg.apply(lambda s: s.sum() / len(s))

4.37 ms ± 255 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


Lo que importa es el **orden de magnitud**: los métodos *built-in* son típicamente varias veces más rápidos que las lambdas, porque evitan llamar a Python por cada grupo.

| Necesito… | Prefiere |
|---|---|
| Estadística estándar | `"sum"`, `"mean"`, `"nunique"`, `"quantile"`… como string |
| Valor de grupo en cada fila | `transform("...")` con string |
| Quedarme con grupos | máscara con `transform` antes que `filter` |
| Top‑N | `sort_values` + `groupby().head(n)` |
| Lógica con varias columnas | crear columnas auxiliares vectorizadas **antes** del `groupby` |

In [31]:
# Ejemplo del último punto: % de pedidos con flete > 20% del precio, por categoría.
# ❌ lambda con varias columnas -> necesitaría apply
# ✅ columna auxiliar + agregación built-in
(items.assign(flete_caro=items["freight_value"] > 0.2 * items["price"])
      .groupby("categoria")["flete_caro"].mean()
      .mul(100).round(1)
      .sort_values(ascending=False)
      .head())

categoria
books_general_interest      100.0
furniture_bedroom           100.0
costruction_tools_garden    100.0
small_appliances            100.0
furniture_living_room       100.0
Name: flete_caro, dtype: float64

---
# ✍️ Ejercicios

Variables disponibles: `orders`, `items` (con `categoria` y `total`), `payments`, `products`, `customers`, `cat_en`.

### Ejercicio 1 · Agregaciones con nombre ⭐

Con `payments`, crea `por_metodo` indexado por `payment_type` con estas columnas, **en este orden**:

| columna | definición |
|---|---|
| `n_pedidos` | número de pedidos **distintos** |
| `monto_total` | suma de `payment_value` |
| `ticket_medio` | media de `payment_value` |
| `cuotas_mediana` | mediana de `payment_installments` |
| `pct_monto` | `monto_total` como % del monto total de todos los métodos |

Ordénalo por `monto_total` descendente.

In [32]:
# TU CÓDIGO AQUÍ
por_metodo = ...

In [33]:
# Comprobación
if pendiente(por_metodo):
  print("⏳ Ejercicio 1 pendiente: resuélvelo en la celda de arriba")
else:
  assert list(por_metodo.columns) == ["n_pedidos", "monto_total", "ticket_medio", "cuotas_mediana", "pct_monto"]
  assert por_metodo.index[0] == "credit_card"
  assert por_metodo["monto_total"].is_monotonic_decreasing
  assert np.isclose(por_metodo["pct_monto"].sum(), 100)
  assert por_metodo.loc["credit_card", "n_pedidos"] == 240
  assert np.isclose(por_metodo.loc["boleto", "ticket_medio"], 106.7835, atol=0.01)
  print("✅ Ejercicio 1 correcto")

⏳ Ejercicio 1 pendiente: resuélvelo en la celda de arriba


### Ejercicio 2 · `transform` dentro del pedido ⭐⭐

Sobre `items`:

1. Crea `it2` con las columnas nuevas:
   - `pct_del_pedido`: `price` de cada ítem como % del `price` total de su pedido.
   - `n_items_pedido`: número de ítems del pedido.
   - `es_principal`: `True` en el ítem de mayor `price` de cada pedido (si hay empate, todos los empatados).
2. `n_dominantes`: entre pedidos con **más de un ítem**, ¿en cuántos el ítem principal representa **más del 70%** del precio del pedido? Cuenta **pedidos**, no ítems.

In [34]:
# TU CÓDIGO AQUÍ
it2 = ...
n_dominantes = ...

In [35]:
# Comprobación
if pendiente(it2, n_dominantes):
  print("⏳ Ejercicio 2 pendiente: resuélvelo en la celda de arriba")
else:
  assert len(it2) == len(items)
  assert {"pct_del_pedido", "n_items_pedido", "es_principal"} <= set(it2.columns)
  assert np.allclose(it2.groupby("order_id")["pct_del_pedido"].sum(), 100)
  assert it2.groupby("order_id")["es_principal"].any().all()
  assert n_dominantes == 2, f"n_dominantes = {n_dominantes}"
  print("✅ Ejercicio 2 correcto")

⏳ Ejercicio 2 pendiente: resuélvelo en la celda de arriba


### Ejercicio 3 · `filter` vs máscara ⭐⭐

1. `items_cat_grandes`: ítems de las categorías que tienen **al menos 10 ítems** y un precio medio **mayor a 100**. Resuélvelo con `groupby(...).filter(...)`.
2. `items_cat_grandes_v2`: el mismo resultado sin `filter`, usando `transform`.
3. `cats_grandes`: lista ordenada alfabéticamente de las categorías que sobrevivieron.

In [36]:
# TU CÓDIGO AQUÍ
items_cat_grandes = ...
items_cat_grandes_v2 = ...
cats_grandes = ...

In [37]:
# Comprobación
if pendiente(items_cat_grandes, items_cat_grandes_v2, cats_grandes):
  print("⏳ Ejercicio 3 pendiente: resuélvelo en la celda de arriba")
else:
  assert items_cat_grandes.sort_index().equals(items_cat_grandes_v2.sort_index())
  assert cats_grandes == ["auto", "baby", "stationery", "watches_gifts"], cats_grandes
  assert len(items_cat_grandes) == 52
  print("✅ Ejercicio 3 correcto")

⏳ Ejercicio 3 pendiente: resuélvelo en la celda de arriba


### Ejercicio 4 · Top‑N y `nth` ⭐⭐

1. `top3_estado`: DataFrame con las **3 categorías con más ingreso** (`price`) para cada estado de **vendedor**. Columnas: `estado_vendedor`, `categoria`, `ingreso`, `rank` (1, 2, 3). Sin `apply`.
   Necesitas cargar `olist_sellers_dataset.csv` (sólo las columnas que uses).
2. `segunda_sp`: el nombre de la **segunda** categoría con más ingreso en SP.

In [38]:
# TU CÓDIGO AQUÍ
top3_estado = ...
segunda_sp = ...

In [39]:
# Comprobación
if pendiente(top3_estado, segunda_sp):
  print("⏳ Ejercicio 4 pendiente: resuélvelo en la celda de arriba")
else:
  assert list(top3_estado.columns) == ["estado_vendedor", "categoria", "ingreso", "rank"]
  assert top3_estado.groupby("estado_vendedor", observed=True)["rank"].max().le(3).all()
  assert top3_estado.groupby("estado_vendedor", observed=True)["ingreso"].apply(lambda s: s.is_monotonic_decreasing).all()
  assert len(top3_estado) == 26, len(top3_estado)
  assert segunda_sp == "sports_leisure", segunda_sp
  print("✅ Ejercicio 4 correcto")

⏳ Ejercicio 4 pendiente: resuélvelo en la celda de arriba


### Ejercicio 5 · Reto: desempeño logístico por estado ⭐⭐⭐

Para pedidos **entregados** (`order_status == "delivered"` y con fecha de entrega), construye `logistica` indexado por estado del **cliente**, con:

| columna | definición |
|---|---|
| `pedidos` | número de pedidos entregados |
| `dias_entrega_med` | mediana de días entre compra y entrega |
| `pct_tarde` | % de pedidos entregados **después** de la fecha estimada |
| `retraso_medio_tarde` | días promedio de retraso **sólo** entre los que llegaron tarde (`NaN` si ninguno) |

Conserva sólo estados con **al menos 10** pedidos y ordena por `pct_tarde` descendente.

Pistas: calcula columnas auxiliares vectorizadas **antes** del `groupby`; para `retraso_medio_tarde` piensa en qué hace `mean` con `NaN`.

In [40]:
# TU CÓDIGO AQUÍ
logistica = ...

In [41]:
# Comprobación
if pendiente(logistica):
  print("⏳ Ejercicio 5 pendiente: resuélvelo en la celda de arriba")
else:
  assert list(logistica.columns) == ["pedidos", "dias_entrega_med", "pct_tarde", "retraso_medio_tarde"]
  assert (logistica["pedidos"] >= 10).all()
  assert logistica["pct_tarde"].is_monotonic_decreasing
  assert list(logistica.index.astype(str)) == ["BA", "RS", "RJ", "PR", "SP", "MG", "SC"], list(logistica.index.astype(str))
  assert np.isclose(logistica.loc["SP", "dias_entrega_med"], 7.754583333333334)
  assert np.isclose(logistica.loc["RJ", "pct_tarde"], 10.5263, atol=0.01)
  print("✅ Ejercicio 5 correcto")

⏳ Ejercicio 5 pendiente: resuélvelo en la celda de arriba


---
# ✅ Soluciones

### Solución 1

In [42]:
por_metodo = (
  payments.groupby("payment_type", observed=True)
  .agg(
    n_pedidos=("order_id", "nunique"),
    monto_total=("payment_value", "sum"),
    ticket_medio=("payment_value", "mean"),
    cuotas_mediana=("payment_installments", "median"),
  )
  .assign(pct_monto=lambda d: d["monto_total"] / d["monto_total"].sum() * 100)
  .sort_values("monto_total", ascending=False)
)
por_metodo.round(2)

,n_pedidos,monto_total,ticket_medio,cuotas_mediana,pct_monto
payment_type,,,,,
credit_card,240,36767.90,152.56,3.0,85.23
boleto,54,5766.31,106.78,1.0,13.37
voucher,11,507.55,46.14,1.0,1.18
debit_card,3,98.65,32.88,1.0,0.23


*Notas:* `nunique` sobre `order_id` evita contar dos veces un pedido pagado en varias partes con el mismo método. `pct_monto` se calcula **después** del `agg` con `assign`, porque necesita el total de todos los grupos.

### Solución 2

In [43]:
gp = items.groupby("order_id")["price"]
it2 = items.assign(
  pct_del_pedido=items["price"] / gp.transform("sum") * 100,
  n_items_pedido=gp.transform("size"),
  es_principal=items["price"].eq(gp.transform("max")),
)

principales = it2[it2["es_principal"] & (it2["n_items_pedido"] > 1)]
n_dominantes = principales.loc[principales["pct_del_pedido"] > 70, "order_id"].nunique()
n_dominantes

2

*Nota:* contamos con `nunique` sobre `order_id` porque un pedido con empate tendría varios ítems "principales". Aquí ninguno de esos empates supera el 70% (dos ítems iguales suman como máximo 50% cada uno), pero contar pedidos explícitamente hace el código correcto aunque cambien los datos.

### Solución 3

In [44]:
gcat = items.groupby("categoria")

items_cat_grandes = gcat.filter(lambda d: len(d) >= 10 and d["price"].mean() > 100)

mask = (gcat["price"].transform("size") >= 10) & (gcat["price"].transform("mean") > 100)
items_cat_grandes_v2 = items[mask]

cats_grandes = sorted(items_cat_grandes["categoria"].unique())
cats_grandes, len(items_cat_grandes)

(['auto', 'baby', 'stationery', 'watches_gifts'], 52)

*Nota:* ojo con los ítems sin categoría: `filter` con `dropna=True` (default) **descarta** ese grupo, y en la versión con `transform` la máscara queda en `NaN` → `False`. Ambas coinciden, pero es buena costumbre verificarlo con un `assert`, como en la comprobación.

### Solución 4

In [45]:
sellers = pd.read_csv(
  DATA / "olist_sellers_dataset.csv",
  usecols=["seller_id", "seller_state"],
  dtype={"seller_state": "category"},
)

top3_estado = (
  items.assign(estado_vendedor=items["seller_id"].map(sellers.set_index("seller_id")["seller_state"]))
  .groupby(["estado_vendedor", "categoria"], observed=True, as_index=False)
  .agg(ingreso=("price", "sum"))
  .sort_values(["estado_vendedor", "ingreso"], ascending=[True, False])
  .groupby("estado_vendedor", observed=True)
  .head(3)
  .assign(rank=lambda d: d.groupby("estado_vendedor", observed=True).cumcount() + 1)
  .reset_index(drop=True)
)

segunda_sp = top3_estado.loc[(top3_estado["estado_vendedor"] == "SP") & (top3_estado["rank"] == 2), "categoria"].item()
segunda_sp, top3_estado.head(6)

('sports_leisure',
   estado_vendedor                 categoria  ingreso  rank
 0              BA                cool_stuff   229.99     1
 1              DF     computers_accessories   319.00     1
 2              DF  fashion_bags_accessories   199.90     2
 3              ES              garden_tools   297.00     1
 4              GO                housewares   179.00     1
 5              GO                      baby   172.33     2)

*Notas:* dos `groupby` encadenados: el primero **agrega** (estado × categoría), el segundo hace el **top‑N** sobre ese resumen. `cumcount() + 1` numera dentro del grupo respetando el orden actual. Alternativa para `segunda_sp`: `...groupby("estado_vendedor").nth(1)`.

### Solución 5

In [46]:
entregados = (
  orders[orders["order_status"].eq("delivered") & orders["order_delivered_customer_date"].notna()]
  .merge(customers, on="customer_id")
)

entregados = entregados.assign(
  dias_entrega=(entregados["order_delivered_customer_date"]
                - entregados["order_purchase_timestamp"]).dt.total_seconds() / 86400,
  retraso=(entregados["order_delivered_customer_date"]
           - entregados["order_estimated_delivery_date"]).dt.total_seconds() / 86400,
)
entregados = entregados.assign(
  tarde=entregados["retraso"] > 0,
  retraso_si_tarde=entregados["retraso"].where(entregados["retraso"] > 0),   # NaN si llegó a tiempo
)

logistica = (
  entregados.groupby("customer_state", observed=True)
  .agg(
    pedidos=("order_id", "size"),
    dias_entrega_med=("dias_entrega", "median"),
    pct_tarde=("tarde", "mean"),
    retraso_medio_tarde=("retraso_si_tarde", "mean"),
  )
  .assign(pct_tarde=lambda d: d["pct_tarde"] * 100)
  .query("pedidos >= 10")
  .sort_values("pct_tarde", ascending=False)
)
logistica.round(2)

,pedidos,dias_entrega_med,pct_tarde,retraso_medio_tarde
customer_state,,,,
BA,11,17.14,18.18,11.19
RS,19,15.38,15.79,7.84
RJ,38,11.45,10.53,12.59
PR,18,9.23,5.56,1.68
SP,127,7.75,4.72,6.32
MG,32,9.67,3.12,0.97
SC,11,14.37,0.00,NaN


*Notas clave:*

- **`where`** convierte en `NaN` los retrasos de pedidos a tiempo; como `mean` ignora `NaN`, `retraso_medio_tarde` queda calculado **sólo** sobre los tardíos, sin `apply` ni `filter`.
- La media de una columna booleana es la proporción de `True`: el truco más útil para porcentajes.
- `query("pedidos >= 10")` filtra el resumen después de agregar (adelanto del Módulo 08).

---
## Resumen del módulo

- `agg` → una fila por grupo; `transform` → mismas filas; `filter` → subconjunto; `apply` → lo que devuelvas.
- Usa **agregaciones con nombre**: `agg(nombre=("col", "func"))`.
- Revisa `dropna` (los `NaN` en la llave desaparecen) y `observed` con categorías; comprueba que los totales cuadren.
- `transform` para % del grupo, z-scores, rellenos y marcas como "es el máximo".
- pandas 3: `apply` ya no ve las columnas de agrupación.
- Top‑N: `sort_values` + `groupby().head(n)`; numerar con `cumcount`.
- `pd.Grouper(key=..., freq="ME")` para agrupar por periodos.
- Rendimiento: funciones *built-in* como string y columnas auxiliares **antes** del `groupby`.

**Siguiente:** Módulo 04 · Reshape: `pivot_table`, `melt`, `stack`/`unstack`, `crosstab` y `explode`.